In [ ]:
import os
import sys
if os.getcwd().endswith("notebooks"):
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_validate

from src.data_loader import load_data
from src.model import get_baseline_models
from src.preprocessing import split_data

In [ ]:
df = load_data("data/creditcard.csv")

X_train, X_val, X_test, y_train, y_val, y_test = split_data(
    df,
    test_size=0.20,
    validation_size=0.20,
    random_state=42,
)

print(X_train.shape, X_val.shape, X_test.shape)
print("Train fraud rate:", y_train.mean())

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "recall": "recall",
    "precision": "precision",
    "f1": "f1",
    "auprc": "average_precision",
}

results = []

for name, model in get_baseline_models().items():
    print(f"Evaluating {name}...")
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False,
    )
    row = {"model": name}
    for metric in scoring:
        row[metric] = scores[f"test_{metric}"].mean()
        row[f"{metric}_std"] = scores[f"test_{metric}"].std()
    results.append(row)

cv_results = (
    pd.DataFrame(results)
    .sort_values("auprc", ascending=False)
    .reset_index(drop=True)
)

cv_results

In [ ]:
Path("results").mkdir(exist_ok=True)
Path("figures").mkdir(exist_ok=True)

cv_results.to_csv("results/model_comparison.csv", index=False)

plot_df = cv_results.sort_values("auprc", ascending=True)

plt.figure(figsize=(9, 5.8))
plt.barh(plot_df["model"], plot_df["auprc"])
plt.xlabel("Mean 5-Fold CV AUPRC")
plt.ylabel("")
plt.title("Model Comparison")
plt.xlim(0, 1)
plt.tight_layout()
plt.savefig("figures/model_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

print("Selected model:", cv_results.iloc[0]["model"])
print("CV AUPRC:", cv_results.iloc[0]["auprc"])